# DRVI Factor & Cluster Analysis

Downstream evaluation of the interpretation from Notebook A: pseudobulk aggregation at the patient level, top-gene overview per factor, differential analysis of individual Leiden clusters (Cohen's d), and factor-/gene-based UMAPs for visual doublet checking.

## 1. Pseudobulk aggregation (cell to patient level)

*Source: `2_3_a_drvi_pseudobulk.py`*

DRVI Pseudobulk — aggregating factor activity from cell level to patient level

DRVI factors are available per cell, but the outcome (e.g. `classification`)
is per patient. This script summarizes the per-cell values into a single
metric per factor (mean and median), producing a unit x factors matrix that
can be used directly for patient-based downstream analyses (e.g.
classification by outcome).

Some patients have multiple samples at different timepoints (e.g. `m6.1` ..
`m6.4` for ACS patient 6, TP1-TP4; controls like `k9` have only one sample).
`sample_id`/`display_name` therefore encode "patient.timepoint", not the
patient alone (same convention as in `DEG/deg_conditions.ipynb`:
`patient_id = display_name.split(".")[0]`). This script therefore supports
two aggregation levels (`--level`):

  - "sample":  one row per sample (patient x timepoint). Timepoint signal is
               preserved; for repeated-measures analyses `patient_id`
               (included in the metadata) must be considered as a blocking
               factor (see deg_conditions.ipynb).
  - "patient": one row per actual patient, averaged across all
               timepoints/samples. Loses temporal resolution, but delivers
               the pure patient x factors matrix requested by the user (one
               patient = one row).

Usage:
    conda run -n mapra_cytokines python 2_2_b_drvi_pseudobulk.py \
        --embed-input .../visualization/drvi_interpretation/embed.h5ad \
        --output-dir  .../visualization/drvi_interpretation \
        --level patient

In [ ]:
import argparse
import os
import anndata as ad
import pandas as pd

In [ ]:
import argparse
args = argparse.Namespace(
    embed_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation/embed.h5ad",
    summary_csv="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation/interpretation_summary.csv",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    sample_key="sample_id",
    outcome_key="classification",
    factor_scope="active",
    level="patient",
)

In [ ]:
os.makedirs(args.output_dir, exist_ok=True)

### Load data

In [ ]:
print("=== Load embedding ===")
embed = ad.read_h5ad(args.embed_input)
print(f"Embedding: {embed.n_obs} cells x {embed.n_vars} factors")

if args.factor_scope == "active" and os.path.exists(args.summary_csv):
    summary = pd.read_csv(args.summary_csv, header=None, index_col=0).squeeze("columns")
    active_factors = [f.strip() for f in str(summary["active_factors"]).split(",")]
    factors = [f for f in active_factors if f in embed.var_names]
    print(f"Active factors (from {os.path.basename(args.summary_csv)}): {len(factors)}")
else:
    factors = embed.var_names.tolist()
    print(f"All factors: {len(factors)}")

if args.sample_key not in embed.obs.columns:
    raise ValueError(f"Sample key '{args.sample_key}' not present in embed.obs.")

### Derive patient/timepoint from the sample ID

In [ ]:
# Same convention as in DEG/deg_conditions.ipynb: "m6.4" -> patient_id="m6", timepoint="4"
# Samples without a dot (e.g. "k9", single draw) -> timepoint=None.

sample_ids = embed.obs[args.sample_key].astype(str)
split       = sample_ids.str.split(".", n=1, expand=True)
patient_id  = split[0]
timepoint   = split[1] if split.shape[1] > 1 else pd.Series(pd.NA, index=sample_ids.index)

samples_per_patient = pd.DataFrame({"patient_id": patient_id.values, "sample": sample_ids.values}) \
    .drop_duplicates().groupby("patient_id", observed=True).size()
n_patients_multi = (samples_per_patient > 1).sum()
print(f"\n{sample_ids.nunique()} samples -> {patient_id.nunique()} patients "
      f"({n_patients_multi} of which have multiple timepoint samples)")

group_key = args.sample_key if args.level == "sample" else "patient_id"

### Pseudobulk: cell -> unit (sample or patient)

In [ ]:
print(f"\n=== Aggregation at '{args.level}' level (grouping: {group_key}) ===")

latent = embed[:, factors].X
if hasattr(latent, "toarray"):
    latent = latent.toarray()

factor_df = pd.DataFrame(latent, index=embed.obs_names, columns=factors)
factor_df["patient_id"] = patient_id.values
factor_df[args.sample_key] = sample_ids.values

grouped = factor_df.groupby(group_key, observed=True)

mean_matrix   = grouped[factors].mean()
median_matrix = grouped[factors].median()
n_cells       = grouped.size().rename("n_cells")

print(f"Matrix shape ({args.level.capitalize()} x factors): {mean_matrix.shape}")

### Save

In [ ]:
mean_path   = os.path.join(args.output_dir, f"pseudobulk_{args.level}_factor_mean.csv")
median_path = os.path.join(args.output_dir, f"pseudobulk_{args.level}_factor_median.csv")
meta_path   = os.path.join(args.output_dir, f"pseudobulk_{args.level}_metadata.csv")

mean_matrix.to_csv(mean_path)
median_matrix.to_csv(median_path)
print(f"Saved: {os.path.basename(mean_path)}")
print(f"Saved: {os.path.basename(median_path)}")

# Metadata is kept separate (outcome + cell count + timepoint info) so that
# the factor matrices themselves stay purely numeric.
obs_extra = embed.obs.copy()
obs_extra["patient_id"] = patient_id.values
obs_extra[args.sample_key] = sample_ids.values

meta_cols = [c for c in [args.outcome_key, "library"] if c in obs_extra.columns]
metadata = (
    obs_extra[[group_key] + meta_cols]
    .drop_duplicates(subset=group_key)
    .set_index(group_key)
)
metadata = metadata.join(n_cells)

if args.level == "patient":
    n_timepoints = (
        obs_extra[["patient_id", args.sample_key]]
        .drop_duplicates()
        .groupby("patient_id", observed=True)
        .size()
        .rename("n_samples")
    )
    metadata = metadata.join(n_timepoints)
else:
    metadata["patient_id"] = obs_extra.drop_duplicates(subset=group_key).set_index(group_key)["patient_id"]

metadata.to_csv(meta_path)
print(f"Saved: {os.path.basename(meta_path)}")

print(f"\nDone. {mean_matrix.shape[0]} {args.level}s x {mean_matrix.shape[1]} factors.")
print(f"Results in: {args.output_dir}")

Alternative: `level="sample"` for patient×timepoint resolution instead of pure patient level.

In [ ]:
args.level = "sample"
# ... re-run the cells above with this changed level to produce the timepoint-resolved variant.

## 2. Top-gene overview per factor (OOD & IND)

*Source: `2_4_a_drvi_top_genes.py`*

DRVI top-gene overview — top-N genes per (non-vanished) factor with score

Purely local evaluation of the results already produced by 2_2_a/2_2_b
(no model/data reload, no network requests):

  - factor_title_mapping.csv  (2_2_b): var_name -> title/order/vanished
  - ood_scores.csv or ind_scores.csv (2_2_a): genes x title-score matrix

Uses the correct var_name -> title mapping (see 2_2_b_drvi_factor_annotation.py
for the background on this) so that the genes really match the corresponding
factor in the pseudobulk matrix.

Result: top_genes_per_factor.csv in long format
    factor, direction, rank, gene, score

Usage:
    conda run -n mapra_cytokines python 2_2_c_drvi_top_genes.py --n-top-genes 10

In [ ]:
import argparse
import os
import pandas as pd

In [ ]:
import argparse
args = argparse.Namespace(
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    score_key="ood",
    n_top_genes=10,
)

In [ ]:
mapping_path = os.path.join(args.output_dir, "factor_title_mapping.csv")
scores_path  = os.path.join(args.output_dir, f"{args.score_key}_scores.csv")

if not os.path.exists(mapping_path):
    raise FileNotFoundError(f"{mapping_path} not found — run 2_2_b_drvi_factor_annotation.py first.")
if not os.path.exists(scores_path):
    raise FileNotFoundError(f"{scores_path} not found — run 2_2_a_drvi_interpretation.py first.")

dim_map = pd.read_csv(mapping_path, index_col=0)
for col in ["vanished", "vanished_positive_direction", "vanished_negative_direction"]:
    dim_map[col] = dim_map[col].astype(bool)

scores = pd.read_csv(scores_path, index_col=0)

active_factors = dim_map.index[~dim_map["vanished"]].tolist()
print(f"Active factors: {len(active_factors)}")

### Top-N genes per factor direction

In [ ]:
rows = []
for factor in active_factors:
    title = dim_map.loc[factor, "title"]
    for direction, vanished_col in [("+", "vanished_positive_direction"),
                                     ("-", "vanished_negative_direction")]:
        if dim_map.loc[factor, vanished_col]:
            continue
        col = f"{title}{direction}"
        if col not in scores.columns:
            print(f"  WARNING: column '{col}' for factor {factor} not found in {os.path.basename(scores_path)} — skipped.")
            continue
        top = scores[col].nlargest(args.n_top_genes)
        for rank, (gene, score) in enumerate(top.items(), start=1):
            rows.append({
                "factor": factor,
                "direction": direction,
                "rank": rank,
                "gene": gene,
                "score": score,
            })

top_genes_df = pd.DataFrame(rows)

out_path = os.path.join(args.output_dir, f"top_genes_per_factor_{args.score_key}.csv")
top_genes_df.to_csv(out_path, index=False)
print(f"\nSaved: {os.path.basename(out_path)}  ({top_genes_df.shape[0]} rows, "
      f"{top_genes_df.groupby(['factor', 'direction']).ngroups} factor directions)")

# Short console preview
preview_factors = active_factors[:3]
for factor in preview_factors:
    sub = top_genes_df[top_genes_df["factor"] == factor]
    for direction in ["+", "-"]:
        d = sub[sub["direction"] == direction]
        if d.empty:
            continue
        genes_str = ", ".join(f"{g} ({s:.2f})" for g, s in zip(d["gene"], d["score"]))
        print(f"  {factor}{direction}: {genes_str}")

Run the same block a second time with `score_key="ind"` to compare both score types (OOD = artificial perturbation, IND = natural cell distribution).

In [ ]:
args.score_key = "ind"
# ... re-run the cells above

## 3. Cluster differential analysis (Cohen's d)

*Source: `2_5_a_drvi_cluster_analysis.py`*

DRVI cluster differential analysis — which factors drive a Leiden cluster?

For a given Leiden cluster (default: cluster 13, res=1.0, from
data/data_for_practicum_post_integration.h5ad), the mean score within the
cluster is compared to the mean of all other cells for each DRVI factor. The
factors are sorted by absolute difference to see which dimension
distinguishes the cluster most strongly from all other cells.

Usage:
    conda run -n mapra_cytokines python 2_5_a_drvi_cluster_analysis.py --cluster 13

In [ ]:
import argparse
import os
import anndata as ad
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
import argparse
args = argparse.Namespace(
    input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    leiden_key="drvi_leiden",
    embedding_key="X_drvi",
    cluster="13",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    mapping_csv="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation/factor_title_mapping.csv",
)

In [ ]:
os.makedirs(args.output_dir, exist_ok=True)

### Load data (only obs + obsm needed, no X)

In [ ]:
print("=== Load data ===")
adata = ad.read_h5ad(args.input, backed="r")

if args.leiden_key not in adata.obs.columns:
    raise ValueError(f"'{args.leiden_key}' not present in obs. Available: {list(adata.obs.columns)}")

cluster_labels = adata.obs[args.leiden_key].astype(str)
if args.cluster not in cluster_labels.unique():
    raise ValueError(f"Cluster '{args.cluster}' not present in '{args.leiden_key}'. "
                     f"Available: {sorted(cluster_labels.unique(), key=lambda x: int(x) if x.isdigit() else x)}")

latent = np.asarray(adata.obsm[args.embedding_key])
n_dims = latent.shape[1]
dim_names = [f"DR{i+1}" for i in range(n_dims)]

mask_in  = (cluster_labels == args.cluster).values
mask_out = ~mask_in
print(f"Cluster {args.cluster}: {mask_in.sum()} cells  |  Rest: {mask_out.sum()} cells")

### Per dimension: mean in cluster vs. rest, difference + context

In [ ]:
# A raw difference alone doesn't say whether the signal is large relative to
# the dimension's natural spread. So additionally:
#   - std per group (context for the difference)
#   - Cohen's d (difference standardized by the pooled spread)
#   - Welch's t-test (robust with very unequal group sizes:
#     1100 vs. 108404 cells -> a classic Student's t-test assuming equal
#     variance would not be appropriate here)

x_in  = latent[mask_in]
x_out = latent[mask_out]

mean_in  = x_in.mean(axis=0)
mean_out = x_out.mean(axis=0)
std_in   = x_in.std(axis=0, ddof=1)
std_out  = x_out.std(axis=0, ddof=1)
diff     = mean_in - mean_out

n_in, n_out = mask_in.sum(), mask_out.sum()
pooled_std = np.sqrt(
    ((n_in - 1) * std_in**2 + (n_out - 1) * std_out**2) / (n_in + n_out - 2)
)
cohens_d = diff / pooled_std

t_stat, p_val = stats.ttest_ind(x_in, x_out, axis=0, equal_var=False)

result = pd.DataFrame({
    "factor": dim_names,
    "mean_in_cluster": mean_in,
    "mean_rest": mean_out,
    "std_in_cluster": std_in,
    "std_rest": std_out,
    "diff": diff,
    "abs_diff": np.abs(diff),
    "cohens_d": cohens_d,
    "abs_cohens_d": np.abs(cohens_d),
    "welch_t": t_stat,
    "p_value": p_val,
}).sort_values("abs_cohens_d", ascending=False).reset_index(drop=True)

if os.path.exists(args.mapping_csv):
    dim_map = pd.read_csv(args.mapping_csv, index_col=0)
    dim_map["vanished"] = dim_map["vanished"].astype(bool)
    result = result.merge(dim_map[["vanished"]], left_on="factor", right_index=True, how="left")
else:
    result["vanished"] = pd.NA

out_path = os.path.join(args.output_dir, f"cluster{args.cluster}_factor_diff.csv")
result.to_csv(out_path, index=False)
print(f"\nSaved: {os.path.basename(out_path)}  ({result.shape[0]} factors)")

print("\n=== Top 10 factors by |Cohen's d| (difference standardized by spread) ===")
print(result.head(10)[["factor", "vanished", "diff", "std_in_cluster", "std_rest",
                       "cohens_d", "p_value"]].to_string(index=False))

top = result.iloc[0]
print(f"\nLargest effect: {top['factor']}  Cohen's d={top['cohens_d']:+.2f}  "
      f"(diff={top['diff']:+.3f}, std_cluster={top['std_in_cluster']:.3f}, "
      f"std_rest={top['std_rest']:.3f}, p={top['p_value']:.1e}, vanished={top['vanished']})")

print("\nFor comparison — top 5 by raw |difference| (without spread context):")
print(result.sort_values("abs_diff", ascending=False).head(5)[["factor", "diff", "cohens_d"]].to_string(index=False))

print(f"\nDone. Results in: {args.output_dir}")

## 4. Factor-gene UMAPs (doublet check)

*Source: `2_6_a_drvi_factor_gene_umaps.py`*

DRVI factor-gene UMAPs — gene expression of the top genes of selected factor
directions on the DRVI UMAP, e.g. to visually check clusters for doublet
signatures (simultaneous expression of markers from different lineages).

Uses the already existing top_genes_per_factor_*.csv (from 2_2_c/2_4_a) and
embed.h5ad (UMAP) — no model reload needed.

Usage:
    conda run -n mapra_cytokines python 2_5_b_drvi_factor_gene_umaps.py \
        --factor-dirs DR7+,DR22-,DR21+,DR35+,DR32+,DR56-,DR48+,DR39-,DR9-,DR4+,DR17+

In [ ]:
import argparse
import os
import re
import warnings
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    drvi_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    factor_dirs="DR7+,DR22-,DR21+,DR35+,DR32+,DR56-,DR48+,DR39-,DR9-,DR4+,DR17+",
    score_key="ind",
    n_genes=4,
    extra_genes="CLEC4C,AXL,GZMB",
    gene_prefixes="",
    out_name="doublet_check_selected_factors.png",
    color_map="YlOrBr",
)

In [ ]:
factor_dirs = [f.strip() for f in args.factor_dirs.split(",") if f.strip()]
parsed = []
for fd in factor_dirs:
    m = re.match(r"^(DR\d+)\s*([+-])$", fd)
    if not m:
        raise ValueError(f"Could not parse '{fd}' (expected e.g. 'DR7+').")
    parsed.append((m.group(1), m.group(2)))
print(f"Factor directions: {parsed}")

embed_path = os.path.join(args.output_dir, "embed.h5ad")
embed = sc.read_h5ad(embed_path)

if parsed:
    genes_path = os.path.join(args.output_dir, f"top_genes_per_factor_{args.score_key}.csv")
    genes_df = pd.read_csv(genes_path)

print("=== Load data (gene expression) ===")
adata = sc.read_h5ad(args.drvi_input)
adata.X = adata.layers["log1p_norm"]

umap_df = pd.DataFrame(embed.obsm["X_umap"], index=embed.obs_names)
adata.obsm["X_drvi_umap"] = umap_df.loc[adata.obs_names].values

sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

genes, labels = [], []
for factor, direction in parsed:
    sub = genes_df[(genes_df["factor"] == factor) & (genes_df["direction"] == direction)]
    if sub.empty:
        print(f"  {factor}{direction}: no top genes found — skipped.")
        continue
    top = sub.sort_values("rank").head(args.n_genes)
    found_any = False
    for gene in top["gene"]:
        if gene in adata.var_names and gene not in genes:
            genes.append(gene)
            labels.append(f"{gene} ({factor}{direction})")
            found_any = True
    if not found_any:
        print(f"  {factor}{direction}: no genes found in the dataset — skipped.")

for gene in [g.strip() for g in args.extra_genes.split(",") if g.strip()]:
    if gene not in adata.var_names:
        print(f"  {gene}: not found in the dataset — skipped.")
    elif gene in genes:
        print(f"  {gene}: already in the list — skipped.")
    else:
        genes.append(gene)
        labels.append(gene)

for prefix in [p.strip() for p in args.gene_prefixes.split(",") if p.strip()]:
    matches = sorted(g for g in adata.var_names if g.startswith(prefix))
    if not matches:
        print(f"  Prefix '{prefix}': no matching genes found.")
        continue
    n_added = 0
    for gene in matches:
        if gene not in genes:
            genes.append(gene)
            labels.append(gene)
            n_added += 1
    print(f"  Prefix '{prefix}': {len(matches)} genes found, {n_added} newly added.")

if not genes:
    raise RuntimeError("No genes found — nothing to plot.")

print(f"\n{len(genes)} genes total: {genes}")

fig = sc.pl.embedding(
    adata, "X_drvi_umap", color=genes, show=False, return_fig=True,
    ncols=4, title=labels, color_map=args.color_map, vmax="p99.5",
)
out = os.path.join(args.output_dir, args.out_name)
fig.savefig(out, bbox_inches="tight", dpi=120)
plt.close("all")
print(f"\nSaved: {out}")

Example of a pure gene-prefix search (independent of factors), e.g. all immunoglobulin genes:

In [ ]:
args.factor_dirs = ""
args.extra_genes = ""
args.gene_prefixes = "IGH,IGL"
args.out_name = "umap_igh_igl_genes.png"
# ... re-run the cells above